# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we display all record sets defined in the dataset, along with their associated fields and columns. Each is referenced by its `@id`.

In [ ]:
# Explore record sets, fields, and columns
from pprint import pprint

if hasattr(metadata, 'record_sets'):
    print(f"Number of record sets: {len(metadata.record_sets)}\n")
    for record_set in metadata.record_sets:
        print(f"RecordSet: {record_set.id}")
        print(f"  name: {record_set.name}")
        print(f"  description: {record_set.description if hasattr(record_set, 'description') else 'None'}")
        # List fields
        if hasattr(record_set, 'fields'):
            print(f"  Fields for {record_set.name}:")
            for field in record_set.fields:
                print(f"    Field: {field.id}")
                print(f"      label: {field.label if hasattr(field, 'label') else 'None'} | dataType: {field.data_type if hasattr(field, 'data_type') else 'None'}")
        # List columns and underlying file objects
        if hasattr(record_set, 'columns'):
            print(f"  Columns for {record_set.name}:")
            for column in record_set.columns:
                print(f"    Column: {column.id}")
                if hasattr(column, 'source'):
                    print(f"      source: {column.source.id if hasattr(column.source, 'id') else column.source}")
        print("")
else:
    print("No record sets found in dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

First, we identify the available record set `@id`s and choose one for demonstration. In this example, we extract all record sets. Use the outputs from the previous cell to select the right record set and field IDs for your analysis.

In [ ]:
# Extract data from each record set
# Identify the available record set IDs
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = [rs.id for rs in metadata.record_sets]
print("Record set @ids found:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set '{record_set_id}' with {len(dataframes[record_set_id])} rows and columns: {list(dataframes[record_set_id].columns)[:8]}")  # show up to 8 columns

# For demonstration, print columns of the first record set loaded (if any)
if record_sets:
    first_rs = record_sets[0]
    print(f"\nColumns for {first_rs}: {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we perform EDA on a numeric field from the first record set. Please ensure to reference fields by their `@id` as observed above.

Change the variable assignments below to match the available field `@id`s for proper EDA.

In [ ]:
# Example EDA on a numeric field
# Ensure you select the correct field @id for a numeric field
# For demonstration, we attempt to select the first field that is float/integer

# Select the record set and field
current_record_set = None
numeric_field = None
group_field = None

for rs in getattr(metadata, 'record_sets', []):
    df = dataframes[rs.id]
    # Look for numeric fields by @id (float or int or their likely names)
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            # Try to use a numeric-looking field
            field_id = f.id
            if field_id in df.columns:
                # Heuristic: check dtype
                col_dtype = df[field_id].dtype
                if pd.api.types.is_numeric_dtype(col_dtype):
                    numeric_field = field_id
                    current_record_set = rs.id
                    break
        # Try to pick a group field
        for f in rs.fields:
            field_id = f.id
            if field_id in df.columns and pd.api.types.is_object_dtype(df[field_id].dtype):
                group_field = field_id
                break
    if numeric_field:
        break

if current_record_set and numeric_field:
    print(f"Using record set '{current_record_set}' and numeric field '{numeric_field}'")
    threshold = (dataframes[current_record_set][numeric_field].mean() if pd.notnull(dataframes[current_record_set][numeric_field].mean()) else 0)
    print(f"Using mean as threshold: {threshold:.4f}")
    # Filter
    filtered_df = dataframes[current_record_set][dataframes[current_record_set][numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.4f} (n = {len(filtered_df)})")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if one exists
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("Could not identify a numeric field for EDA. Please update the field ID variable assignments above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We provide an example visualization for the selected numeric field.

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

if current_record_set and numeric_field:
    sns.histplot(dataframes[current_record_set][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of field {numeric_field} in record set {current_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in dataframes[current_record_set].columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[current_record_set].dropna(subset=[group_field, numeric_field]))
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR²-compliant dataset, using `mlcroissant`, referencing all entity `@id`s.

- We loaded the dataset schema and metadata.
- Explored the available record set and field structures using their canonical IDs.
- Performed basic data filtering, normalization, and grouping on a selected numeric field.
- Visualized dataset distributions.

You can now extend this notebook with more advanced analyses and visualizations tailored to your specific research or data processing needs.